In [5]:
import os 
import glob
import tqdm
import numpy as np
from skimage import measure
import pandas as pd
import nibabel as nb

from meld_graph.paths import MELD_DATA_PATH

Setting MELD_DATA_PATH to /home/mathilde/Documents/projects/MELD_graph/261022_MELD_graph_v2.2.7_H101_gate/meld_data
Setting BASE_PATH to /home/mathilde/Documents/projects/MELD_graph/261022_MELD_graph_v2.2.7_H101_gate/meld_data/output/preprocessed_surf_data
Setting EXPERIMENT_PATH to /home/mathilde/Documents/projects/MELD_graph/261022_MELD_graph_v2.2.7_H101_gate/meld_data/models
Setting FS_SUBJECTS_PATH to /home/mathilde/Documents/projects/MELD_graph/261022_MELD_graph_v2.2.7_H101_gate/meld_data/output/fs_outputs
Setting BASE_PATH to /home/mathilde/Documents/projects/MELD_graph/261022_MELD_graph_v2.2.7_H101_gate/meld_data/meld_params


In [107]:
def load_nii(file):
    ''' Load a nifti file and return the data as a numpy array. If the file cannot be loaded, return None.'''
    try: 
        data = nb.load(file).get_fdata()
    except: 
        data = None
    return data


def cluster_and_threshold(mask, min_area_threshold=100):
    """
    Cluster a 3D binary mask into connected components and remove clusters
    smaller than a given voxel count.

    Args:
        mask (ndarray): 3D numpy array (binary mask, 0 = background, 1 = foreground).
        min_area_threshold (int): Minimum number of voxels required to keep a cluster.

    Returns:
        ndarray: 3D array of same shape as mask, where:
                 - 0 = background
                 - 1..N = filtered connected components
    """

    # Step 1: Label connected components
    labels = measure.label(mask, connectivity=1)  # 1 = 6-connectivity in 3D

    # Step 2: Initialize filtered label map (same shape as mask)
    filtered_labels = np.zeros_like(labels)

    # Step 3: Iterate over each connected component (skip 0 = background)
    component_count = 0
    label_ids = []
    for label_id in range(1, labels.max() + 1):
        component_size = np.sum(labels == label_id)

        # Keep only large enough clusters
        if component_size >= min_area_threshold:
            component_count += 1
            filtered_labels[labels == label_id] = component_count
            label_ids.append(component_count)
    
    return filtered_labels, label_ids

def compute_metrics(pred, labels, min_area_threshold=3):
    """ Compute metrics (e.g. sensitivity, specificity, number of false positive, ..) for a given prediction and labels
    Args: 
        pred: numpy array containing the predicted lesion mask
        labels: numpy array containing the label lesion mask (binary)
    Returns:
        a dictionary containing the results of the evaluation at subject level, with the following columns:
            - id: subject id
            - group: patient or control depending on the presence of lesions in the label
            - lesion_size: size of the lesion in the label (number of voxels)
            - subject_detected: whether the subject's lesion is detected if patient or no clusters are predicted if control (True/False)
            - n_lesion_cls: number of lesion clusters in the label
            - n_lesion_cls_detected: number of lesion clusters in the label that are detected by the prediction
            - median_dice_cls_detected: median dice score of the detected lesion clusters
            - n_pred_cls: number of predicted lesion clusters
            - n_pred_cls_fp: number of predicted lesion clusters that are false positives (not overlapping with any lesion cluster in the label)
            - n_pred_cls_tp: number of predicted lesion clusters that are true positives (overlapping with at least one lesion cluster in the label)
            - voxel_dice: dice score between the predicted lesion mask and the label lesion mask at voxel level
            - voxel_iou: iou score between the predicted lesion mask and the label lesion mask at voxel level    """

    # predictions are already thresholded and clustered
    pred_clustered = pred.copy()
    pred_cls = np.unique(pred)
    pred_cls = [cl for cl in pred_cls if cl!=0 and cl<100]
    for pred_cl in pred_cls:
        pred_clustered[pred_clustered==pred_cl*100] = pred_cl
    n_pred_cls = len(pred_cls)
    del pred

    # lesion size
    if labels is None:
        lesion_size = 0
    else:
        lesion_size = labels.sum()

    # if no lesions, subject is control, and subject is detected if no predicted clusters
    if lesion_size==0:
        group = 'control'
        subject_detected = n_pred_cls==0
        n_lesion_cls = None
        n_lesion_cls_detected = None
        median_dice_cls_detected = None
        n_pred_cls_fp = n_pred_cls
        voxel_dice = None
        voxel_iou = None
    else:
        group = 'patient'
    
        # cluster and threshold the lesion mask, discard clusters smaller than 3 voxels, and get number of lesion clusters
        lesion_clustered, lesion_cls  = cluster_and_threshold(labels, min_area_threshold=3)
        n_lesion_cls = len(lesion_cls)
        del labels

        # for each lesion cluster, find if detected and compute dice
        lesion_cls_detected = np.zeros(n_lesion_cls).astype(bool)
        lesion_cls_dice = np.zeros(n_lesion_cls)
        pred_cls_detected = []
        for i, lesion_cl in enumerate(lesion_cls):
            lesion_cl_mask = (lesion_clustered == lesion_cl)
            overlap_cls = np.unique(pred_clustered[lesion_cl_mask])
            overlap_cls = [x for x in overlap_cls if x!=0 ]

            if len(overlap_cls) == 0:
                lesion_cls_detected[i] = False
                lesion_cls_dice[i] = 0
            else:
                lesion_cls_detected[i] = True
                
                # if lesion is detected by multiple predicted cluster, take the one with the highest dice score
                dices_cl = []
                for pid in overlap_cls:
                    pred_cl_mask = (pred_clustered == pid)
                    dice_cl = 2 * np.sum(lesion_cl_mask & pred_cl_mask) / (np.sum(lesion_cl_mask) + np.sum(pred_cl_mask))
                    dices_cl.append(dice_cl)
                    pred_cls_detected.append(pid)
                lesion_cls_dice[i] = max(dices_cl)
                
        # find number of detected lesion and median dice score in lesion detected
        n_lesion_cls_detected = lesion_cls_detected.sum()
        median_dice_cls_detected = np.median(lesion_cls_dice[lesion_cls_detected])
        
        # find if subject detected and voxel dice score and iou
        subject_detected = lesion_cls_detected.sum()>0
        voxel_dice = 2 * np.sum((lesion_clustered>0) & (pred_clustered>0)) / (np.sum(lesion_clustered>0) + np.sum(pred_clustered>0))
        voxel_iou = np.sum((lesion_clustered>0) & (pred_clustered>0)) / np.sum((lesion_clustered>0) | (pred_clustered>0))
        
        # find number of false positives predicted clusters
        pred_cls_detected = np.unique(pred_cls_detected)
        n_pred_cls_detected = len(pred_cls_detected)
        n_pred_cls_fp = n_pred_cls - n_pred_cls_detected
        
    metrics = {
        "group": group,
        "lesion_size": lesion_size,
        "subject_detected": subject_detected,
        "n_lesion_cls": n_lesion_cls,
        "n_lesion_cls_detected": n_lesion_cls_detected,
        "median_dice_cls_detected": median_dice_cls_detected,
        "n_pred_cls": n_pred_cls,
        "n_pred_cls_fp": n_pred_cls_fp,
        "n_pred_cls_tp": n_pred_cls-n_pred_cls_fp,
        "voxel_dice": voxel_dice, 
        "voxel_iou":voxel_iou,
    }
    return metrics

def evaluate_prediction_subject_level(df, output_file=None, overwrite=False, min_area_threshold=3):
    ''' Evaluate performances predictions at subject level by comparing the predicted lesion mask with the label lesion mask.
    
    Args:
        df: dataframe containing columns:
            - 'id' with the subject id
            - 'label_file' with the path to the ground truth file
            - 'pred_file' with the path to the prediction file 
        output_file: path to the output csv file to save the results (optional). If not provided, the results will be saved in the pred_folder with the name 'summary_evaluation_subject_level.csv'
    
    Returns:
        a pandas dataframe containing the results of the evaluation at subject level, with the following columns:
            - id: subject id
            - group: patient or control depending on the presence of lesions in the label
            - lesion_size: size of the lesion in the label (number of voxels)
            - subject_detected: whether the subject's lesion is detected if patient or no clusters are predicted if control (True/False)
            - n_lesion_cls: number of lesion clusters in the label
            - n_lesion_cls_detected: number of lesion clusters in the label that are detected by the prediction
            - median_dice_cls_detected: median dice score of the detected lesion clusters
            - n_pred_cls: number of predicted lesion clusters
            - n_pred_cls_fp: number of predicted lesion clusters that are false positives (not overlapping with any lesion cluster in the label)
            - n_pred_cls_tp: number of predicted lesion clusters that are true positives (overlapping with at least one lesion cluster in the label)
            - voxel_dice: dice score between the predicted lesion mask and the label lesion mask at voxel level
            - voxel_iou: iou score between the predicted lesion mask and the label lesion mask at voxel level   
    '''
    
    # do not overwrite existing results except if overwrite is set to True
    if not output_file is None:
        if (os.path.isfile(output_file)) and (overwrite==False):
            print(f'File {output_file} already exists. Data will not be saved. Please delete or set overwrite to True')
            return None      
    
    # compute results prediction per predicted clusters
    results = []
    for i, row in tqdm.tqdm(df.iterrows(), total=len(df)) : 

        subject = row['id']
        # print(subject)

        # load labels
        if not row['label_file'] is None:
            labels = load_nii(row['label_file'])
            labels = (labels>0).astype(int)
        else:
            labels = None
        
        pred = load_nii(row['pred_file'])
        # pred = (pred>0).astype(int)

        metrics = compute_metrics(pred, labels, min_area_threshold=min_area_threshold)
        metrics['id'] = subject
        # update results with subject info
        results.append(metrics)

    df_res = pd.DataFrame(results) 
    
    # merge with files
    df = df_res.merge(df, on='id')
    
    # save results analysis
    if not output_file is None:
        df.to_csv(output_file, index=False)
        print(f'Summary of results saved at {output_file}')

    return df

In [88]:
# find input, label and pred folder
list_ids = '/home/mathilde/Documents/projects/MELD_graph/261022_MELD_graph_v2.2.7_H101_gate/meld_data/list_subjects_H101.txt'

subjects=np.array(np.loadtxt(list_ids, dtype='str', ndmin=1))

print('Number subjects: ', len(subjects))

Number subjects:  170


In [ ]:
# copy coregistered lesion mask from MELD FE v0.2.1

# import shutil
# bids_folder = '/home/mathilde/Documents/RDS/MELD_FE/DATA/MELD_H101/QC_new/BIDS_v0.1'

# for subject in subjects:
#     bids_id = ''.join(subject.split('_'))
#     bids_id = bids_id.replace('3T','')
#     bids_id = bids_id.replace('FCD', 'P')
#     print(bids_id)
#     bids_lesions = glob.glob(os.path.join(bids_folder, f'sub-{bids_id}', 'anat', f'sub-{bids_id}_*_space-orig_mod-T1w_label-lesion_mask.nii.gz'))
#     if len(bids_lesions)>0:
#         bids_lesions = bids_lesions[0]
#         copy_file = os.path.join(MELD_DATA_PATH, 'input', subject, 'lesion_mask', os.path.basename(bids_lesions))
#         shutil.copy(bids_lesions,copy_file)

In [110]:
with_gate = True

In [111]:
# get input, label and predictions

print(f'Find files for {len(subjects)} subjects')
files = []
for subject in subjects:
    data_files = {'id': subject}
    # get lesion mask file
    label_file = glob.glob(os.path.join(MELD_DATA_PATH, 'input', subject, 'lesion_mask', '*lesion_mask.nii.gz'))
    if len(label_file) == 1:
        data_files['label_file'] = label_file[0]
    elif len(label_file) >1:
        print(f'Multiple label files found for {subject}. Taking first one')
        data_files['label_file'] = label_file[0]
    else:
        data_files['label_file'] = None
    # get prediction file
    if with_gate:
        pred_file = os.path.join(MELD_DATA_PATH, 'output', 'predictions_reports_gate', subject, 'predictions', 'prediction.nii.gz')
    else:
        pred_file = os.path.join(MELD_DATA_PATH, 'output', 'predictions_reports', subject, 'predictions', 'prediction.nii.gz')
    if os.path.isfile(pred_file):
        data_files['pred_file'] = pred_file
    else:
        data_files['pred_file'] = None
    files.append(data_files)

df_files = pd.DataFrame(files)
df_files = df_files.dropna(subset=['pred_file']).reset_index(drop=True)
print(f'Number of subjects prediction files: {len(df_files)}')

Find files for 170 subjects
Number of subjects prediction files: 170


In [112]:
# launch evaluation
if with_gate :
  summary_eval_file = os.path.join(MELD_DATA_PATH,'summary_evaluation_subject_level_gate.csv')
else:
  summary_eval_file = os.path.join(MELD_DATA_PATH,'summary_evaluation_subject_level_nogate.csv')
  
df = evaluate_prediction_subject_level(df_files,
                                      output_file = summary_eval_file,
                                      overwrite = True, 
                                      min_area_threshold=3,
                                        )
df.head()

 52%|█████▏    | 88/170 [01:41<01:30,  1.11s/it]/home/mathilde/anaconda3/envs/meld_graph/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3474: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/mathilde/anaconda3/envs/meld_graph/lib/python3.9/site-packages/numpy/core/_methods.py:189: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
 54%|█████▎    | 91/170 [01:46<02:07,  1.62s/it]/home/mathilde/anaconda3/envs/meld_graph/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3474: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/mathilde/anaconda3/envs/meld_graph/lib/python3.9/site-packages/numpy/core/_methods.py:189: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
 55%|█████▌    | 94/170 [01:50<01:34,  1.24s/it]/home/mathilde/anaconda3/envs/meld_graph/lib/python3.9/site-packages/numpy/core/fromnumeri

Summary of results saved at /home/mathilde/Documents/projects/MELD_graph/261022_MELD_graph_v2.2.7_H101_gate/meld_data/summary_evaluation_subject_level_gate.csv


,group,lesion_size,subject_detected,n_lesion_cls,n_lesion_cls_detected,median_dice_cls_detected,n_pred_cls,n_pred_cls_fp,n_pred_cls_tp,voxel_dice,voxel_iou,id,label_file,pred_file
0,control,0,True,NaN,NaN,NaN,0,0,0,NaN,NaN,MELD_H101_3T_C_00002,None,/home/mathilde/Documents/projects/MELD_graph/2...
1,control,0,True,NaN,NaN,NaN,0,0,0,NaN,NaN,MELD_H101_3T_C_00005,None,/home/mathilde/Documents/projects/MELD_graph/2...
2,control,0,True,NaN,NaN,NaN,0,0,0,NaN,NaN,MELD_H101_3T_C_00007,None,/home/mathilde/Documents/projects/MELD_graph/2...
3,control,0,True,NaN,NaN,NaN,0,0,0,NaN,NaN,MELD_H101_3T_C_00008,None,/home/mathilde/Documents/projects/MELD_graph/2...
4,control,0,True,NaN,NaN,NaN,0,0,0,NaN,NaN,MELD_H101_3T_C_00011,None,/home/mathilde/Documents/projects/MELD_graph/2...


In [95]:
from tabulate import tabulate
def summary_table(df, label_var='group',):
    ''' summarise the results of evaluation at subject level in a table with sensitivity, recall, precision and F1-score for each category of the label variable (e.g. group)'''
    
    print(f'\nbreakdown in {label_var}:\n')
    df = df[df[label_var].notna()]
    summary_data = []
    for label in set(df[label_var]):
        tot = len(df[df[label_var] == label])
        
        detect = len(df[(df[label_var] == label) & (df['subject_detected'] == True)])
        if tot ==0:
            summary_data.append([label, tot, np.nan, np.nan, np.nan])
        else:
            
            sensitivity = round((detect / tot) * 100, 1)
            mean_fp = df[df[label_var] == label]['n_pred_cls_fp'].quantile([0.25, 0.5, 0.75]).values
            ppv = round(np.sum(df[(df[label_var]==label)]['n_pred_cls_tp']) / (np.sum(df[(df[label_var]==label)]['n_pred_cls_fp']) + np.sum(df[(df[label_var]==label)]['n_pred_cls_tp'])),2)
            if ('n_lesion_cls_detected' in df.keys()) & ('n_lesion_cls' in df.keys()):
                recall = round(np.mean(df[(df[label_var]==label)]['n_lesion_cls_detected']/df[(df[label_var]==label)]['n_lesion_cls']),2)
                f1 = round(2*(ppv*recall)/(ppv+recall),2)
            else:
                recall = 'not computed'
                f1 = 'not computed'
            if 'voxel_dice' in df.keys():
                mean_subject_dice = round(df[(df[label_var] == label) & (df['subject_detected'] == True)]['voxel_dice'].mean(),2)
            else:
                mean_subject_dice = 'not computed'
            if 'n_lesion_cls' in df.keys():
                mean_n_lesion_cls = round(np.mean(df[(df[label_var]==label)]['n_lesion_cls']),2)
            else:
                mean_n_lesion_cls = 'not computed'
            mean_n_pred_cls = round(np.mean(df[(df[label_var]==label)]['n_pred_cls']),2)
            summary_data.append([label, tot, sensitivity, recall, ppv, f1, mean_fp ,mean_n_lesion_cls, mean_n_pred_cls, mean_subject_dice])

    # Sort summary_data by the "Number" column (index 1)
    summary_data = sorted(summary_data, key=lambda x: x[1], reverse=True)  # descending order (largest first)
    # Create and print the summary table
    headers = ["Category", "Number", "Sensitivity\nSpecificity",  "Recall", "Precision\n(PPV)", "F1-score", "FPs\nIQR [25,50,75]", "Average\nlesions", "Average\npredictions",  "Dice\nScore"]
    summary_table = tabulate(summary_data, headers=headers, tablefmt="grid")
    # print(summary_table)
    return summary_table

In [113]:
# compare results gating no gating 

models = {'with_gate':os.path.join(MELD_DATA_PATH,'summary_evaluation_subject_level_gate.csv'), 
         'no_gate':os.path.join(MELD_DATA_PATH,'summary_evaluation_subject_level_nogate.csv')}

for model in models.keys():
    df = pd.read_csv(models[model])
    print(f'Summary table for model {model}')
    print(summary_table(df, label_var='group',))

Summary table for model with_gate

breakdown in group:

+------------+----------+---------------+----------+-------------+------------+------------------+-----------+---------------+---------+
| Category   |   Number |   Sensitivity |   Recall |   Precision |   F1-score | FPs              |   Average |       Average |    Dice |
|            |          |   Specificity |          |       (PPV) |            | IQR [25,50,75]   |   lesions |   predictions |   Score |
+============+==========+===============+==========+=============+============+==================+===========+===============+=========+
| control    |       85 |          96.5 |   nan    |        0    |     nan    | [0. 0. 0.]       |    nan    |          0.04 |   nan   |
+------------+----------+---------------+----------+-------------+------------+------------------+-----------+---------------+---------+
| patient    |       85 |          49.4 |     0.49 |        0.96 |       0.65 | [0. 0. 0.]       |      1.04 |          0.

In [116]:
df = pd.read_csv(models['with_gate'])
df.head(100)

,group,lesion_size,subject_detected,n_lesion_cls,n_lesion_cls_detected,median_dice_cls_detected,n_pred_cls,n_pred_cls_fp,n_pred_cls_tp,voxel_dice,voxel_iou,id,label_file,pred_file
0,control,0,True,NaN,NaN,NaN,0,0,0,NaN,NaN,MELD_H101_3T_C_00002,NaN,/home/mathilde/Documents/projects/MELD_graph/2...
1,control,0,True,NaN,NaN,NaN,0,0,0,NaN,NaN,MELD_H101_3T_C_00005,NaN,/home/mathilde/Documents/projects/MELD_graph/2...
2,control,0,True,NaN,NaN,NaN,0,0,0,NaN,NaN,MELD_H101_3T_C_00007,NaN,/home/mathilde/Documents/projects/MELD_graph/2...
3,control,0,True,NaN,NaN,NaN,0,0,0,NaN,NaN,MELD_H101_3T_C_00008,NaN,/home/mathilde/Documents/projects/MELD_graph/2...
4,control,0,True,NaN,NaN,NaN,0,0,0,NaN,NaN,MELD_H101_3T_C_00011,NaN,/home/mathilde/Documents/projects/MELD_graph/2...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,patient,1427,False,1.0,0.0,NaN,0,0,0,0.000000,0.000000,MELD_H101_3T_FCD_00116,/home/mathilde/Documents/projects/MELD_graph/2...,/home/mathilde/Documents/projects/MELD_graph/2...
96,patient,2731,False,1.0,0.0,NaN,0,0,0,0.000000,0.000000,MELD_H101_3T_FCD_00066,/home/mathilde/Documents/projects/MELD_graph/2...,/home/mathilde/Documents/projects/MELD_graph/2...
97,patient,8530,True,1.0,1.0,0.505339,1,0,1,0.505339,0.338096,MELD_H101_3T_FCD_00090,/home/mathilde/Documents/projects/MELD_graph/2...,/home/mathilde/Documents/projects/MELD_graph/2...
98,patient,98755,True,1.0,1.0,0.070736,2,0,2,0.062467,0.032241,MELD_H101_3T_FCD_00063,/home/mathilde/Documents/projects/MELD_graph/2...,/home/mathilde/Documents/projects/MELD_graph/2...
